# Análisis de Denuncias por Violencia Contra la Mujer en Perú (2018 - 2026)

Este notebook tiene como objetivo procesar y analizar un dataset de denuncias por violencia contra la mujer en Perú, recopilado entre enero de 2018 y marzo de 2026. El flujo de trabajo incluye la carga de datos, transformación, limpieza y almacenamiento de los datos procesados, preparando la información para futuros análisis o visualizaciones.

In [3]:
import pandas as pd


## 1. Carga de Datos Raw

En esta sección, cargamos el dataset original de denuncias. Este archivo contiene información detallada sobre las denuncias, incluyendo la modalidad de violencia, la ubicación (departamento, provincia, distrito), el año, el mes y la cantidad de casos.

In [4]:
denuncias = pd.read_csv('../data/raw/DATASET_Viol_Cont_Muj_Ene 2018 a Mar 2026.csv')

## 2. Exploración Inicial de Datos

Aquí mostramos las primeras filas del DataFrame `denuncias` para tener una visión rápida de la estructura del dataset cargado, los tipos de datos y los valores que contiene. Esto nos ayuda a entender el formato de los datos crudos antes de cualquier transformación.

In [5]:
denuncias

,MODALIDAD,PROV_HECHO,DIST_HECHO,UBIGEO_HECHO,AÑO,MES,DPTO_HECHO,CANTIDAD
0,VIOLENCIA ECONOMICA O PATRIMONIAL,HUANCAYO,HUANCAYO,120101,2020,3,JUNIN,5
1,VIOLENCIA PSICOLOGICA,ICA,OCUCAJE,110104,2020,5,ICA,7
2,VIOLENCIA PSICOLOGICA,ILO,ILO,180301,2020,11,MOQUEGUA,83
3,VIOLENCIA FISICA,LA CONVENCION,MARANURA,80904,2023,12,CUSCO,1
4,VIOLENCIA PSICOLOGICA,CUTERVO,CUTERVO,60601,2018,2,CAJAMARCA,7
...,...,...,...,...,...,...,...,...
194420,VIOLENCIA FISICA,AREQUIPA,YURA,40128,2020,6,AREQUIPA,3
194421,VIOLENCIA PSICOLOGICA,TRUJILLO,TRUJILLO,130101,2018,9,LA LIBERTAD,111
194422,VIOLENCIA FISICA,UTCUBAMBA,JAMALCA,10705,2021,8,AMAZONAS,4
194423,VIOLENCIA PSICOLOGICA,MARISCAL CACERES,CAMPANILLA,220602,2025,8,SAN MARTIN,1


## 3. Preprocesamiento y Transformación de Datos

Esta sección se encarga de transformar el dataset original para facilitar el análisis. Los pasos clave incluyen:

*   **Separar modalidades en columnas**: Utilizamos `pivot_table` para convertir las diferentes modalidades de violencia (ej. violencia psicológica, física, económica) de filas a columnas, permitiendo un análisis por tipo de violencia.
*   **Cálculo del total de denuncias**: Se agrega una columna 'total' que suma todas las modalidades de violencia por cada fila (año, ubigeo, etc.) para obtener el total de denuncias por cada combinación de atributos.
*   **Normalización de nombres de columnas**: Se transforman los nombres de las columnas a minúsculas y se reemplazan los espacios por guiones bajos para una mayor consistencia y facilidad de uso.
*   **Formato de `ubigeo_hecho`**: La columna `ubigeo_hecho` (código geográfico) se convierte a tipo string y se rellena con ceros a la izquierda hasta asegurar que tenga 6 dígitos, lo cual es crucial para la identificación geográfica precisa.

In [6]:
# separar modalidad en columnas
denuncias_modalidad = denuncias.pivot_table(
    index=["AÑO", "UBIGEO_HECHO", "DPTO_HECHO", "PROV_HECHO", "DIST_HECHO"],
    columns="MODALIDAD",
    values="CANTIDAD",
    aggfunc="sum",
    fill_value=0
).reset_index()

denuncias_modalidad.columns.name = None

columnas_modalidad = [
    col for col in denuncias_modalidad.columns
    if col not in ["AÑO", "UBIGEO_HECHO", "DPTO_HECHO", "PROV_HECHO", "DIST_HECHO"]
]

denuncias_modalidad["total"] = (
    denuncias_modalidad[columnas_modalidad].sum(axis=1)
)

denuncias_modalidad.columns = denuncias_modalidad.columns.str.lower().str.replace(" ", "_", regex=False)
# convertir ubigeo_hecho a string y rellenar con ceros a la izquierda
denuncias_modalidad["ubigeo_hecho"] = denuncias_modalidad["ubigeo_hecho"].astype(str).str.zfill(6)

denuncias_modalidad

,año,ubigeo_hecho,dpto_hecho,prov_hecho,dist_hecho,amenaza_grave,coaccion_grave,maltrato_sin_lesion,violencia_economica_o_patrimonial,violencia_fisica,violencia_fisica_y_psicologica,violencia_psicologica,violencia_sexual,total
0,2018,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,0,0,0,9,3,290,265,0,567
1,2018,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,0,0,0,0,0,1,0,0,1
2,2018,010104,AMAZONAS,CHACHAPOYAS,CHETO,0,0,0,0,0,1,1,0,2
3,2018,010105,AMAZONAS,CHACHAPOYAS,CHILIQUIN,0,0,0,0,1,1,1,0,3
4,2018,010107,AMAZONAS,CHACHAPOYAS,GRANADA,0,0,0,0,0,1,2,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13263,2026,250301,UCAYALI,PADRE ABAD,PADRE ABAD,0,0,0,1,19,0,24,0,44
13264,2026,250302,UCAYALI,PADRE ABAD,IRAZOLA,0,0,0,0,6,0,5,0,11
13265,2026,250303,UCAYALI,PADRE ABAD,CURIMANA,0,0,0,0,2,0,2,0,4
13266,2026,250304,UCAYALI,PADRE ABAD,NESHUYA,0,0,0,0,3,0,6,0,9


## 4. Renombrado de Columnas

Para mejorar la legibilidad y estandarización del DataFrame procesado, se renombran algunas columnas clave. Esto facilita la comprensión de los datos y su manipulación en análisis posteriores.

In [7]:
# renombrar columnas
denuncias_modalidad.rename(columns={
    "año": "anio",
    "ubigeo_hecho": "ubigeo",
    "dpto_hecho": "departamento",
    "prov_hecho": "provincia",
    "dist_hecho": "distrito"
}, inplace=True)

## 5. Guardar Datos Procesados

Una vez que los datos han sido transformados y limpiados, se guardan en un nuevo archivo CSV en la carpeta `../data/processed`. Primero, se verifica si la carpeta existe y, si no, se crea. Esto asegura que los datos preprocesados estén disponibles para su uso en otras etapas del pipeline, como la modelización o la visualización, sin necesidad de repetir los pasos de preprocesamiento.

In [8]:
import os

output_dir = '../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

denuncias_modalidad.to_csv(os.path.join(output_dir, 'denuncias_modalidad.csv'), index=False)